# 05 · Delegação entre modelos

Modelos maiores raciocinam melhor, mas custam mais latência (e, em outros contextos, mais
dinheiro). Uma estratégia recorrente em sistemas de agentes é a **orquestração hierárquica**: um
modelo pequeno e rápido conduz o laço e resolve as etapas simples, **delegando** apenas as
sub-tarefas que exigem raciocínio mais profundo a um modelo maior.

O ponto conceitual relevante é que o modelo maior é encapsulado como **uma ferramenta** qualquer.
Do ponto de vista do laço, `consultar_especialista` não se distingue de uma calculadora: o
orquestrador decide quando acioná-la, exatamente como decide quando calcular. Toda a infraestrutura
permanece no catálogo da NVIDIA — apenas variamos o modelo entre o orquestrador e o especialista.


## Dependências

In [1]:
!pip install -q openai

## Configuração do cliente

Empregamos a biblioteca `openai` apontando para o endpoint **gratuito da NVIDIA**
(`https://integrate.api.nvidia.com/v1`), que expõe uma interface compatível com a API da OpenAI.
A célula seguinte concentra os parâmetros de acesso: endpoint, modelo e credencial.

Para obter a credencial: criar conta em https://build.nvidia.com e gerar uma API key (prefixo
`nvapi-`). No Google Colab, a chave pode ser guardada uma única vez em *Secrets* (ícone de chave na
barra lateral) com o nome `NVIDIA_API_KEY`; a célula a seguir a detecta automaticamente. O modelo
padrão (`meta/llama-3.1-8b-instruct`) responde em cerca de um segundo por chamada e suporta
*function calling* de forma confiável.


In [2]:
import os
import json
from getpass import getpass
from datetime import datetime
from openai import OpenAI

def _obter_api_key():
    try:                                  # 1) Colab Secrets (icone de chave a esquerda)
        from google.colab import userdata
        chave = userdata.get("NVIDIA_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    return os.environ.get("NVIDIA_API_KEY") or getpass("API key (nvapi-...): ")  # 2) env var ou 3) manual

BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL    = "meta/llama-3.1-8b-instruct"
API_KEY  = _obter_api_key()

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("Cliente configurado:", BASE_URL, "| modelo:", MODEL)


Cliente configurado: https://integrate.api.nvidia.com/v1 | modelo: meta/llama-3.1-8b-instruct


In [3]:
# Modelo grande, acionado apenas sob demanda pela ferramenta de delegacao
MODEL_ESPECIALISTA = "meta/llama-3.1-70b-instruct"
print("Orquestrador:", MODEL, "| Especialista:", MODEL_ESPECIALISTA)


Orquestrador: meta/llama-3.1-8b-instruct | Especialista: meta/llama-3.1-70b-instruct


## As ferramentas

`calculate` é resolvida localmente. `consultar_especialista` encaminha uma pergunta ao modelo
maior e devolve sua resposta — uma chamada à API aninhada dentro da execução de uma ferramenta.


In [4]:
def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"


def consultar_especialista(pergunta: str) -> str:
    # Delega ao modelo maior; acionada apenas para raciocinio conceitual
    r = client.chat.completions.create(
        model=MODEL_ESPECIALISTA,
        messages=[
            {"role":"system","content":"Responda de forma rigorosa e concisa."},
            {"role":"user","content":pergunta},
        ],
    )
    return r.choices[0].message.content


TOOL_FUNCTIONS = {"calculate": calculate, "consultar_especialista": consultar_especialista}

TOOLS = [
    {"type":"function","function":{
        "name":"calculate",
        "description":"Avalia uma expressao matematica simples.",
        "parameters":{"type":"object",
            "properties":{"expression":{"type":"string","description":"ex.: '847 * 0.15'"}},
            "required":["expression"]}}},
    {"type":"function","function":{
        "name":"consultar_especialista",
        "description":"Encaminha uma pergunta conceitual ou de raciocinio complexo a um modelo maior e mais capaz. Use para explicacoes, justificativas e analises; NAO use para aritmetica.",
        "parameters":{"type":"object",
            "properties":{"pergunta":{"type":"string","description":"A pergunta a delegar"}},
            "required":["pergunta"]}}},
]


## O laço

O laço é o mesmo dos notebooks anteriores. A política de roteamento — o que o orquestrador resolve
sozinho e o que delega — é instruída pelo *system prompt*.


In [5]:
def run_agent(task, model=MODEL, verbose=True):
    messages = [
        {"role":"system","content":(
            "Voce e um orquestrador. Resolva aritmetica com a ferramenta calculate. "
            "Para qualquer pergunta conceitual, de explicacao ou de raciocinio, use "
            "consultar_especialista. Quando tiver a resposta final, responda sem chamar ferramentas."
        )},
        {"role":"user","content":task},
    ]
    while True:
        r = client.chat.completions.create(model=model, messages=messages, tools=TOOLS, tool_choice="auto")
        msg = r.choices[0].message
        if msg.tool_calls and len(msg.tool_calls) > 1:
            msg.tool_calls = msg.tool_calls[:1]   # endpoint aceita 1 tool call por turno
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if verbose:
                etiqueta = "(delegando ao especialista)" if name == "consultar_especialista" else ""
                print(f"   -> {name}({args}) {etiqueta}")
            fn = TOOL_FUNCTIONS.get(name)
            result = fn(**args) if fn else f"Ferramenta desconhecida: {name}"
            messages.append({"role":"tool","tool_call_id":tc.id,"content":str(result)})


## Execução

Para isolar o comportamento de roteamento, executamos duas tarefas distintas com o **mesmo**
agente. A primeira é puramente aritmética: o orquestrador a resolve localmente, sem delegar. A
segunda é conceitual: o orquestrador a encaminha ao especialista. Observe, nos logs, qual
ferramenta é acionada em cada caso.


In [6]:
print("=== (a) Tarefa aritmetica — resolvida pelo orquestrador ===")
print(run_agent("Quanto e 15% de 847?"))


=== (a) Tarefa aritmetica — resolvida pelo orquestrador ===
   -> calculate({'expression': '847 * 0.15'}) 
A resposta é 127.05.


In [7]:
print("=== (b) Tarefa conceitual — delegada ao especialista ===")
print(run_agent("Explique, com rigor e em duas frases, por que a recursao requer um caso base."))


=== (b) Tarefa conceitual — delegada ao especialista ===
   -> consultar_especialista({'pergunta': 'A recursão requer um caso base porque ela precisa de uma condição para parar a chamada recursiva e evitar um loop infinito. Isso permite que a recursão se torne uma ferramenta prática e eficaz para resolver problemas complexos.'}) (delegando ao especialista)
A resposta final é: A recursão requer um caso base porque ela precisa de uma condição para parar a chamada recursiva e evitar um loop infinito, garantindo que o processo recursivo termine em algum momento. Isso permite que a recursão se torne uma ferramenta prática e eficaz para resolver problemas complexos.


## O custo da delegação

A chamada ao especialista é a etapa mais lenta do laço, justamente por envolver um modelo maior.
O ganho da arquitetura está em pagar essa latência **apenas quando necessário**: tarefas simples
permanecem rápidas no orquestrador, e a capacidade adicional é invocada sob demanda. A mesma
estrutura generaliza para hierarquias com mais níveis ou especialistas distintos por domínio.


---
Continuação: `06_mcp.ipynb` — exposição e descoberta de ferramentas entre processos via Model
Context Protocol (MCP), desacoplando as ferramentas do código do agente.
